# Read Table -> Establish Generic Silver-level Table

In [0]:
#Imports to run API Data Scraping script

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from datetime import datetime
import logging
import traceback
import json
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *


In [0]:
#parameters that need to be changed between notebooks

silver_level_schema = StructType([
    StructField('mcc_code', IntegerType(), True),
    StructField('description', StringType(), True)
])

schema = silver_level_schema
schema_path = 'financial_transactions_dataset.default'
volume_path = '/Volumes/financial_transactions_dataset/default/financial_transactions_dataset_analytics_volume'
volume_identifier = 'mcc_codes'
silver_table_name = 'silver_mcc_codes_data'
bronze_table_name = 'bronze_mcc_codes_data'

In [0]:
#Set up logger to track pipeline activities

def setup_logging():

    # Ensure logs directory exists before starting logging
    os.makedirs('logs', exist_ok=True)

    logging.basicConfig(
        level = logging.INFO,
        format = '%(asctime)s - %(levelname)s - %(message)s',
        handlers = [logging.FileHandler(f'logs/silver_level_table_run_{datetime.now().strftime('%Y%m%d')}.log'),
                    logging.StreamHandler(sys.stdout)
                   ]
    )

    return logging.getLogger(__name__)

In [0]:
#Check if a table exists in a schema; create an empty table if it does not exist.

def check_or_create_table(logger, silver_table_name, schema_path):

    logger.info(f'Checking to see if {silver_table_name} exists.')
    
    if spark.catalog.tableExists(f'{schema_path}.{silver_table_name}'):
        logger.info("Table exists; table creation skipped.")
        flag = 1
    else:
        logger.info('Table does not exist; table will be created.')
        flag = 0
    
    return flag


In [0]:
#Pull data from bronze-level preliminary table

def read_table_as_df(logger, schema_path, bronze_table_name):

    logger.info(f'Reading data from {schema_path}.{bronze_table_name}')

    df = spark.read.table(f'{schema_path}.{bronze_table_name}')
    total_row_count = df.count()

    logger.info(f'Read {total_row_count} rows from {schema_path}.{bronze_table_name}')
    
    return df


In [0]:
def transpose_dataframe(logger, df):

    logger.info(f'Transposing {schema_path}.{bronze_table_name} table.')

    # Get all column names except metadata columns
    mcc_columns = [col for col in df.columns]

    # Stack all columns into rows
    transposed_df = df.selectExpr(
        f"stack({len(mcc_columns)}, {', '.join([f"'{col}', `{col}`" for col in mcc_columns])}) as (mcc_code, mcc_code_description)"
    )

    return transposed_df


In [0]:
def trim_fields(logger, df):

    logger.info(f'Trimming fields in {schema_path}.{bronze_table_name}')

    trimmed_df = df.withColumn('mcc_code', F.trim(F.col('mcc_code'))).withColumn('mcc_code_description', F.trim(F.col('mcc_code_description')))

    return trimmed_df

In [0]:
#Function to overwrite/append data to silver level table

def create_or_append_to_table(logger, table_exists, df, schema_path, silver_table_name):

    if table_exists == 1:
        logger.info(f'Appending data to {schema_path}')

        df.write.format('delta').mode('append').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    else:

        logger.info(f'Creating table to {schema_path}')

        df.write.format('delta').mode('overwrite').saveAsTable(f'{schema_path}.{silver_table_name}')

        total_row_count = df.count()

        logger.info(f'Wrote {total_row_count} rows to {schema_path}')
    
    return


In [0]:
def silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier):

    schema = silver_level_schema
    schema_path = schema_path
    volume_path = volume_path
    volume_identifier = volume_identifier
    silver_table_name = silver_table_name
    bronze_table_name = bronze_table_name
 
    logger = setup_logging()

    logger.info(f'Initializing ETL on bronze dataframe')

    is_created = check_or_create_table(logger, silver_table_name, schema_path)
    bronze_df = read_table_as_df(logger, schema_path, bronze_table_name)
    transposed_df = transpose_dataframe(logger, bronze_df)
    silver_df = trim_fields(logger, transposed_df)

    create_or_append_to_table(logger, is_created, silver_df, schema_path, silver_table_name)
    
    logger.info(f'Silver level table {silver_table_name} created or appended to successfully.')

    return


In [0]:
silver_level_transformation(silver_level_schema, schema_path, bronze_table_name, silver_table_name, volume_path, volume_identifier)